# Few-Shot Prompting — CodeLlama-7B-Instruct


## 1. Setup


In [ ]:
import os

# --- Colab auto-setup ---
IN_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")
REPO_URL = "https://github.com/MinhTuan2405/IE105.21_SBugDwLLM.git"
BRANCH = "trung"
NOTEBOOK_SUBDIR = "prompt/codellama"

if IN_COLAB:
    REPO_DIR = f"/content/IE105.21_SBugDwLLM"
    if not os.path.exists(REPO_DIR):
        pass
        !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
    os.chdir(os.path.join(REPO_DIR, NOTEBOOK_SUBDIR))
    print(f"Working directory: {os.getcwd()}")

!pip install -q transformers datasets accelerate bitsandbytes
!pip install -q scikit-learn matplotlib seaborn huggingface_hub

from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass

if hf_token:
    login(token=hf_token)
    print("Logged in with HF_TOKEN")
else:
    login()  # Fallback: interactive prompt


In [ ]:
import sys
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from utils.data_loader import load_defect_dataset, build_prompt_for_model, build_examples_for_model, build_error_records, split_error_types
from utils.evaluation import evaluate_predictions, parse_prediction_with_fallback, build_evaluation_payload, compare_saved_results, print_result_comparison

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 2. Load Model


In [ ]:
MODEL_ID = "codellama/CodeLlama-7b-Instruct-hf"
MAX_SEQ_LENGTH = 2048
RESULTS_DIR = "../../docs/codellama_docs"
NUM_EXAMPLES = 4
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model.eval()
print(f"Model loaded. Memory: {model.get_memory_footprint() / 1e9:.2f} GB")


## 3. Load Dataset & Prepare Few-Shot Examples


In [ ]:
train_data, _, test_data = load_defect_dataset()
print(f"Test samples: {len(test_data)}")

few_shot_examples = build_examples_for_model(train_data, "few_shot", seed=42)
print(f"\nFew-shot examples ({len(few_shot_examples)}):")
for i, (code, label) in enumerate(few_shot_examples):
    label_str = "Defective" if label == 1 else "Safe"
    print(f"  Example {i+1}: {label_str} (label={label}), {len(code)} chars")


## 4. Few-Shot Prompting Inference


In [ ]:
# Preview prompt
prompt = build_prompt_for_model(MODEL_ID, test_data[0]["func"], tokenizer=tokenizer, examples=few_shot_examples)
print(prompt[:1000])
print("...")


In [ ]:
device = next(model.parameters()).device
y_true, y_pred, failed_parses = [], [], []
truncated_count = 0


for i, sample in enumerate(tqdm(test_data, desc="Few-Shot Prompting inference")):
    prompt = build_prompt_for_model(MODEL_ID, sample["func"], tokenizer=tokenizer, examples=few_shot_examples)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH)
    full_len = len(tokenizer.encode(prompt))
    if full_len > MAX_SEQ_LENGTH:
        truncated_count += 1

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=5, do_sample=False)

    generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    pred, parse_failed = parse_prediction_with_fallback(generated, sample["target"])

    if parse_failed:
        failed_parses.append({"index": i, "output": generated, "label": int(sample["target"])})

    y_true.append(int(sample["target"]))
    y_pred.append(pred)

print(f"\nFailed to parse: {len(failed_parses)} / {len(test_data)}")
print(f"Truncated prompts: {truncated_count} / {len(test_data)}")
if failed_parses[:5]:
    print("Sample failed:")
    for fp in failed_parses[:5]:
        print(f"  idx={fp['index']}, output='{fp['output']}'")


## 5. Evaluation


In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

errors = build_error_records(test_data, y_true, y_pred, max_chars=200)
false_positives, false_negatives = split_error_types(errors)
payload = build_evaluation_payload(
    failed_parses=failed_parses,
    errors=errors,
    false_positives=false_positives,
    false_negatives=false_negatives,
    truncated_count=truncated_count
)

metrics = evaluate_predictions(
    y_true, y_pred,
    model_name="codellama",
    strategy="few_shot",
    save_dir=RESULTS_DIR,
    extra_data=payload,
)


## 6. Error Analysis


In [ ]:
print(f"Total errors: {len(errors)}")
print(f"False Positives: {len(false_positives)}")
print(f"False Negatives: {len(false_negatives)}")

print("\n--- Sample False Negatives ---")
for e in false_negatives[:3]:
    print(f"  idx={e['index']}: {e['code'][:100]}...")

print("\n--- Sample False Positives ---")
for e in false_positives[:3]:
    print(f"  idx={e['index']}: {e['code'][:100]}...")


## 7. Prompt Template Used


In [ ]:
print("=" * 60)
print("FEW-SHOT PROMPT TEMPLATE")
print("=" * 60)
template = build_prompt_for_model(MODEL_ID, "<CODE_HERE>", tokenizer=tokenizer, examples=few_shot_examples)
print(template)


## 8. Compare All Prompting Strategies

Load tất cả kết quả prompting để so sánh.


In [ ]:
results = compare_saved_results(RESULTS_DIR, "codellama")
if results:
    print("\n" + "="*60)
    print("  ALL RESULTS")
    print("="*60)
    print_result_comparison(results)
